## 1. Install the required packages

In [ ]:
%pip install kagglehub librosa matplotlib ipython opencv-python mediapipe tqdm numpy ffmpeg imageio[ffmpeg] joblib pandas

## 2. Download the CREMA-D dataset from Kaggle

In [ ]:
from pathlib import Path

import kagglehub

# Download latest version
path = kagglehub.dataset_download("orvile/crema-d-emotional-multimodal-dataset")
path = Path(path) / "content" / "CREMA-D"

print("Path to dataset files:", path)

## 3. Extract the landmarks from the videos using MediaPipe and save them as .npy files to the `Landmarks` folder.

In [ ]:
from joblib import Parallel, delayed
from tqdm.auto import tqdm
import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks.python import BaseOptions, vision

INPUT_VIDEO_DIR = path / "VideoFlash"
OUTPUT_DATA_DIR = Path("Landmarks")
OUTPUT_DATA_DIR.mkdir(parents=True, exist_ok=True)

def process_video_worker(video_path):
    npy_path = OUTPUT_DATA_DIR / video_path.with_suffix(".npy").name
    if npy_path.exists():
        return  # Skip already processed files

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print(f"Warning: Could not open video {video_path.name}")
        return

    # Initialize MediaPipe locally inside each worker process
    base_options = BaseOptions(model_asset_path="face_landmarker.task")
    options = vision.FaceLandmarkerOptions(
        base_options=base_options,
        running_mode=vision.RunningMode.VIDEO,
        output_face_blendshapes=False,
        output_facial_transformation_matrixes=False,
        num_faces=1,
    )
    detector = vision.FaceLandmarker.create_from_options(options)

    video_sequence = []
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps <= 0 or np.isnan(fps):
        fps = 30.0

    frame_idx = 0
    timestamp_offset = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret or frame is None:
            break

        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
        current_ts = timestamp_offset + int((frame_idx / fps) * 1000)
        
        results = detector.detect_for_video(mp_image, current_ts)

        if results.face_landmarks:
            frame_landmarks = [[lm.x, lm.y, lm.z] for lm in results.face_landmarks[0]]
            video_sequence.append(frame_landmarks)
        else:
            video_sequence.append(np.zeros((478, 3)))

        frame_idx += 1

    cap.release()

    if video_sequence:
        np_sequence = np.array(video_sequence, dtype=np.float32)
        np.save(npy_path, np_sequence)

# Run across all CPU cores using joblib
video_files = list(INPUT_VIDEO_DIR.glob("*.flv"))
print(f"Extracting landmarks from {len(video_files)} videos using joblib multiprocessing...")

# n_jobs=-1 utilizes all available CPU cores safely on Windows
Parallel(n_jobs=-1)(
    delayed(process_video_worker)(video) 
    for video in tqdm(video_files)
)

print("Landmark extraction complete!")

## 4. Visualize the results on a random sample

In [ ]:
import os
import random
from pathlib import Path
import cv2
import numpy as np
import imageio.v3 as iio
import subprocess
import imageio_ffmpeg

import librosa
import librosa.display
import matplotlib.pyplot as plt
from IPython.display import Audio, Video, display

AUDIO_PATH = path / "AudioWAV"
VIDEO_PATH = path / "VideoFlash"
LANDMARK_PATH = Path("Landmarks")

data = []
for file in list(LANDMARK_PATH.glob("*.npy")):
    parts = file.stem.split("_")
    data.append(
        {
            "ActorID": parts[0],
            "Sentence": parts[1],
            "Emotion": parts[2],
            "Intensity": parts[3],
            "AudioPath": AUDIO_PATH / f"{file.stem}.wav",
            "VideoPath": VIDEO_PATH / f"{file.stem}.flv",
            "LandmarkPath": file,
        }
    )

sample = random.choice(data)
print(
    f"Loading Multi-Modal Data for Actor: {sample['ActorID']} | Emotion: {sample['Emotion']}"
)
input_flv = str(sample["VideoPath"])
wav_path = str(sample["AudioPath"])

# Load audio to get exact duration
y_audio, sr = librosa.load(wav_path, sr=None)
audio_duration = len(y_audio) / sr

# Load video with OpenCV
cap = cv2.VideoCapture(input_flv)

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

# Fallback for invalid FPS reading
if fps <= 0 or np.isnan(fps):
    fps = 30.0

print(f"Video loaded. Resolution: {width}x{height}, FPS: {fps:.2f}")

landmarks_seq = np.load(sample["LandmarkPath"])

frames = []
for landmarks in landmarks_seq:
    ret, frame = cap.read()
    if not ret or frame is None:
        break
    
    # Draw landmarks
    for x, y, _ in landmarks:
        cv2.circle(frame, (int(x * width), int(y * height)), 1, (0, 255, 0), -1)
    
    # OpenCV uses BGR; convert to RGB for imageio
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    frames.append(frame_rgb)

cap.release()

if len(frames) == 0:
    raise RuntimeError(f"Could not read frames from video: {input_flv}")

fps_out = len(frames) / audio_duration

silent_mp4 = "silent.mp4"
final_mp4 = "out_with_audio.mp4"

# Write directly to H.264 MP4 using imageio's internal bindings
iio.imwrite(
    silent_mp4,
    frames,
    fps=fps_out,
    codec="libx264",
    pixelformat="yuv420p"
)

ffmpeg = imageio_ffmpeg.get_ffmpeg_exe()                       # works even if ffmpeg isn't on PATH
subprocess.run([
    ffmpeg, "-y",
    "-i", silent_mp4,
    "-i", wav_path,
    "-c:v", "copy", "-c:a", "aac",
    "-shortest",
    final_mp4
], check=True, capture_output=True)

print(f"Successfully saved {len(frames)} frames to {final_mp4}")

display(Video(final_mp4, embed=True))

y, sr = librosa.load(sample["AudioPath"], sr=None)
display(Audio(data=y, rate=sr))
plt.figure(figsize=(10, 3))
librosa.display.waveshow(y, sr=sr, alpha=0.6)
plt.title(f"Audio Waveform - Emotion: {sample['Emotion']}")
plt.tight_layout()
plt.show()

## 5. Display data about the landmarks

In [ ]:
import pandas as pd

# Directory where .npy files are stored
data_dir = Path("./Landmarks")

# Initialize information lists
file_paths = []
emotion_labels = []
actor_ids = []

# Loop over all .npy files
for file_path in data_dir.glob("*.npy"):
    # Get the filename without the .npy extension
    filename_stem = file_path.stem

    # Split the filename by "_"
    parts = filename_stem.split("_")

    # Extract information
    actor_id = parts[0]
    emotion = parts[2]

    # Append the data to the lists
    file_paths.append(str(file_path))
    emotion_labels.append(emotion)
    actor_ids.append(actor_id)

# Create a DataFrame from the lists
df = pd.DataFrame({
    "file_path": file_paths,
    "emotion": emotion_labels,
    "actor_id": actor_ids
})

# Print out the information
print(f"The number of files: {len(file_paths)}")
print(f"The number of unique actors: {df['actor_id'].nunique()}")
print("\nCounts per emotion:")
print(df["emotion"].value_counts())

df["n_frames"] = df["file_path"].apply(
    lambda p: np.load(p, mmap_mode="r").shape[0]
)

print(f"\nThe least amount of frames in one clip: {df['n_frames'].min()}")
print(f"Median of frames in the clips: {df['n_frames'].median()}")
print(f"The most amount of frames in one clip: {df['n_frames'].max()}")

## 6. Detect outliers and exlcude them from the data

In [ ]:
# How many clips are very short or very long?
print("Very short clips - very long clips:")
print((df["n_frames"] < 10).sum(), "-", (df["n_frames"] > 300).sum())

# 1. Find the extremely short file
short_files = df[df["n_frames"] < 10]
print("\n--- Extremely Short File ---")
print(short_files[["file_path", "n_frames", "emotion", "actor_id"]])

# 2. Find the extremely long file
long_files = df[df["n_frames"] > 300]
print("\n--- Extremely Long File ---")
print(long_files[["file_path", "n_frames", "emotion", "actor_id"]])

In [ ]:
# Exclude the outliers
MIN_FRAMES = 10
MAX_FRAMES = 300

outliers = df[(df["n_frames"] < MIN_FRAMES) | (df["n_frames"] > MAX_FRAMES)]
print("Dropping: ", len(outliers), "clips")

# Create a dataframe without outliers
df_clean = df[(df["n_frames"] >= MIN_FRAMES) & (df["n_frames"] <= MAX_FRAMES)].reset_index(drop=True)
print(len(df), "->", len(df_clean))

## 7. Normalization and resampling for each clip

In [ ]:
def normalize_clip(x):
    """Remove head position and face size from every frame of a landmark clip.

    Each frame is handled independently, so only the shape of the face remains.

    Parameters
    ----------
    x : np.ndarray, shape (N, 478, 3)
        N frames, 478 face landmarks, (x, y, z) coordinates per landmark.

    Returns
    -------
    np.ndarray, shape (N, 478, 3)
        Centered and scaled landmarks.
    """
    # Remove position: subtract each frame's mean landmark, so the face center is at the origin
    # Results in a shape of (N, 1, 3)
    center = np.mean(x, axis=1, keepdims=True)
    x_centered = x - center

    # Remove scale: RMS distance of the landmakrs from the center, one number per frame.
    # Results in a shape of (N, 1, 1)
    scale = np.sqrt(np.mean(x_centered ** 2, axis=(1,2), keepdims=True))
    return x_centered / (scale + 1e-8)  # Guards against division by zero

In [ ]:
def resample_clip(x, num_frames):
    """Resample a (N, 478, 3) clip to num_frames evenly spaced frames.

    If num_frames is None or <= 0, the clip is returned unchanged.
    Returns (clip, idx), where idx are the frame indices used.
    """
    if not num_frames or num_frames <= 0:
        return x, np.arange(len(x))
    idx = np.linspace(0, len(x) - 1, num_frames).round().astype(int)
    return x[idx], idx

In [ ]:
NUM_FRAMES = 64 # Resampling size of each clip (SET TO 0 IF NO RESAMPLING IS WANTED)
emotion_map = {"ANG": 0, "DIS": 1, "FEA": 2, "HAP": 3, "SAD": 4, "NEU": 5}

# Load, normalize and resample each clip
clips = []
for path in df_clean["file_path"]:
    x = np.load(path)           # (N, 478, 3), N(number of frames) varies between clips
    x = normalize_clip(x)       # Remove position and scale per frame

    # Resampling
    x, _ = resample_clip(x, NUM_FRAMES)     # (NUM_FRAMES, 478, 3)
    clips.append(x)

# Build dataset arrays
if NUM_FRAMES:
    X = np.stack(clips)                                 # (n_clips, NUM_FRAMES, 487, 3)
else:
    X = clips                                           # list of (N_i, 478, 3) arrays
y = df_clean["emotion"].map(emotion_map).to_numpy()     # emotion as integer 0-5
groups = df_clean["actor_id"].to_numpy()                # actor IDs

# Sanity checks
print("Dataset structure:")
if NUM_FRAMES:
    print("X shape:", X.shape)
else:
    print(f"X is a list of {len(X)} clips with variable lengths.")
    print(f"First clip shape: {X[0].shape}, Last clip shape: {X[-1].shape}")

print("Emotions: ", y.shape, "Actors: ", groups.shape)
print("\nClips per emotion class:")
print(np.bincount(y))

In [ ]:
# Save normalized and resampled clips if needed
'''

from pathlib import Path

OUT_DIR = Path("Landmarks_normalized")
OUT_DIR_RS = Path(f"Landmarks_normalized_64_{NUM_FRAMES}")
OUT_DIR.mkdir(exist_ok=True)
if NUM_FRAMES:
    OUT_DIR_RS.mkdir(exist_ok=True)

for path in df_clean["file_path"]:
    name = Path(path).name
    x = np.load(path)

    # Save normalized landmarks only
    x = normalize_clip(x).astype(np.float32)
    np.save(OUT_DIR / name, x)

    # Save normalized + resampled (only if resampling is applied)
    if NUM_FRAMES:
        x_rs, _ = resample_clip(x, NUM_FRAMES)
        np.save(OUT_DIR_RS / name, x_rs)

print(len(list(OUT_DIR.glob("*.npy"))))
if NUM_FRAMES:
    print(len(list(OUT_DIR_RS.glob("*.npy"))))   # expect 7440 for both

'''


## 7. Display the result of normalization and resampling

In [ ]:
def render_landmarks(clip, size=400, extent=3.0):
    """Draw each frame of a (N, 478, 3) normalized clip as dots on a black canvas.

    extent: half-width of the visible area in normalized units.
    The face has RMS size 1, so extent=3 leaves some margin.
    """
    frames = []
    for lm in clip:
        canvas = np.zeros((size, size, 3), dtype=np.uint8)
        # map [-extent, extent] -> [0, size]; flip y because image y points down
        px = ((lm[:, 0] / extent + 1) / 2 * size).astype(int)
        py = ((lm[:, 1] / extent + 1) / 2 * size).astype(int)
        for u, v in zip(px, py):
            if 0 <= u < size and 0 <= v < size:
                cv2.circle(canvas, (u, v), 1, (0, 255, 0), -1)
        frames.append(canvas)
    return frames

# Pick a random clip from the processed dataset
i = random.randrange(len(X))
print("Emotion:", df_clean["emotion"].iloc[i], "| Actor:", df_clean["actor_id"].iloc[i],
      "| File:", df_clean["file_path"].iloc[i])

norm_frames = render_landmarks(X[i])

# Raw clip for comparison: same frames, same canvas scheme, but only centered and scaled by a constant
raw = np.load(df_clean["file_path"].iloc[i])
if NUM_FRAMES > 0:
    raw = raw[np.linspace(0, len(raw) - 1, NUM_FRAMES).round().astype(int)]
raw_centered = (raw - raw.mean(axis=(0, 1), keepdims=True)) / raw.std() # one fixed shift/scale for the whole clip
raw_frames = render_landmarks(raw_centered)

# Put raw and normalized side by side
side_by_side = [np.hstack((r, n)) for r, n in zip(raw_frames, norm_frames)]

iio.imwrite("normalized_check.mp4", side_by_side, fps=15, codec="libx264", pixelformat="yuv420p")
display(Video("normalized_check.mp4", embed=True))

In [ ]:
import subprocess
import imageio_ffmpeg

def make_comparison(i, out_file="compare_audio.mp4"):
    """Original video + raw landmarks (left), normalized/resampled landmarks (right), with audio."""
    path = df_clean["file_path"].iloc[i]
    stem = Path(path).stem
    print("Clip:", stem, "| Emotion:", df_clean["emotion"].iloc[i])

    # --- Frame indices used for resampling (same as in the dataset) ---
    raw = np.load(path)                                            # (N, 478, 3)
    norm, idx = resample_clip(normalize_clip(raw), NUM_FRAMES)

    # --- Left panel: original video frames with raw landmarks ---
    cap = cv2.VideoCapture(str(VIDEO_PATH / f"{stem}.flv"))
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    video_frames = []
    while True:
        ret, f = cap.read()
        if not ret:
            break
        video_frames.append(f)
    cap.release()

    left = []
    for k in idx:
        f = video_frames[min(k, len(video_frames) - 1)].copy()     # guard against length mismatch
        for x, y, _ in raw[k]:
            cv2.circle(f, (int(x * w), int(y * h)), 1, (0, 255, 0), -1)
        f = cv2.resize(f, (400, 400))
        left.append(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))

    # --- Right panel: saved normalized + resampled clip on a blank canvas ---
    right = render_landmarks(norm)

    side_by_side = [np.hstack((l, r)) for l, r in zip(left, right)]

    # --- Match video duration to the audio duration ---
    wav_path = AUDIO_PATH / f"{stem}.wav"
    y_audio, sr = librosa.load(wav_path, sr=None)
    duration = len(y_audio) / sr
    if NUM_FRAMES >0:
        fps_out = NUM_FRAMES / duration
    else:
        fps_out = len(norm) / duration

    # --- Write silent video, then add the audio track ---
    iio.imwrite("compare_silent.mp4", side_by_side, fps=fps_out,
                codec="libx264", pixelformat="yuv420p")

    ffmpeg = imageio_ffmpeg.get_ffmpeg_exe()                       # works even if ffmpeg isn't on PATH
    subprocess.run([
        ffmpeg, "-y",
        "-i", "compare_silent.mp4",
        "-i", str(wav_path),
        "-c:v", "copy", "-c:a", "aac",
        "-shortest",
        out_file
    ], check=True, capture_output=True)

    display(Video(out_file, embed=True))

    # Same as your original code: separate audio player + waveform
    display(Audio(data=y_audio, rate=sr))
    plt.figure(figsize=(10, 3))
    librosa.display.waveshow(y_audio, sr=sr, alpha=0.6)
    plt.title(f"Audio Waveform - Emotion: {df_clean['emotion'].iloc[i]}")
    plt.tight_layout()
    plt.show()

# Random clip:
make_comparison(random.randrange(len(df_clean)))

## WavLM Audio Features

In [ ]:
import librosa
import numpy as np
import torch
from tqdm.auto import tqdm
from transformers import Wav2Vec2FeatureExtractor, WavLMModel

MODEL_ID = "microsoft/wavlm-base"
TARGET_SR = 16_000

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_ID)
model = WavLMModel.from_pretrained(MODEL_ID).to(device)
model.eval()

audio_embeddings = []

for record in tqdm(data, desc="Extracting WavLM embeddings", unit="audio"):
    waveform, _ = librosa.load(
        record["AudioPath"],
        sr=TARGET_SR,
        mono=True,
    )
    if waveform.size == 0:
        raise ValueError(f"Empty audio file: {record['AudioPath']}")

    inputs = feature_extractor(
        waveform,
        sampling_rate=TARGET_SR,
        return_tensors="pt",
        return_attention_mask=True,
    )
    inputs = {name: tensor.to(device) for name, tensor in inputs.items()}

    with torch.inference_mode():
        frame_features = model(**inputs).last_hidden_state[0]

    mean = frame_features.mean(dim=0)
    std = frame_features.std(dim=0, unbiased=False)
    audio_embeddings.append(torch.cat((mean, std)).cpu().numpy())

audio_embeddings = np.stack(audio_embeddings)
print("Embedding matrix shape:", audio_embeddings.shape)

#### Save the WavLM embeddings to AudioFeatures folder

In [ ]:
FEATURE_DIR = Path("AudioFeatures")
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

records_for_audio = data[:len(audio_embeddings)]

for record, embedding in zip(records_for_audio, audio_embeddings):
    stem = Path(record["AudioPath"]).stem
    output_path = FEATURE_DIR / f"{stem}.npy"
    np.save(output_path, np.asarray(embedding, dtype=np.float32))

print(f"Saved {len(audio_embeddings)} audio feature files to {FEATURE_DIR.resolve()}")
print("Example:", output_path, "| vector shape:", embedding.shape)